In [8]:
import os
from os.path import join as pjoin
import pandas as pd


In [2]:

lines=[]
for app in os.listdir(pjoin("repair-benchmark","apps")):
    pr_dirs=[pjoin("repair-benchmark","apps",app,"issues",folder) for folder in (os.listdir(pjoin("repair-benchmark","apps",app,"issues"))) if not folder.endswith("_private")]
    for pr in pr_dirs:
        for model in ["codex"]:
            with open(pjoin("repair-benchmark","apps",app,"issues",pr.split("/")[-1],f"run_{model}.txt"),"w") as f:
                lines.append(f"python3 O3_Script_agentic_ais.py --pr {pr} --model {model}\n")

#Create a text file and write all the lines to it
with open("run_all.txt","w") as f:
    f.writelines(lines)

In [ ]:
####################################################################################################
# Rerunning for empty patches
####################################################################################################

empty_patches='''          503646332 2207657543 1154179361 1336005533 1348102534 1701725770 1845281685 1163893375
'''
empty_patches_agent=["claude","codex --codex-model gpt-6-astra","qwen"]
empty_patches_list=[patch.strip().split("_")[-1] for patch in empty_patches.split("\n") if patch.strip()]

lines=[]
for app in os.listdir(pjoin("repair-benchmark","apps")):
    pr_dirs=[pjoin("repair-benchmark","apps",app,"issues",folder) for folder in (os.listdir(pjoin("repair-benchmark","apps",app,"issues"))) if not folder.endswith("_private")]
    for pr in pr_dirs:
        for model in ["claude","codex --codex-model gpt-6-astra","qwen"]:
            with open(pjoin("repair-benchmark","apps",app,"issues",pr.split("/")[-1],f"run_{model}.txt"),"w") as f:
                if pr.split("_")[-1] in empty_patches_list and model in empty_patches_agent:
                    lines.append(f"python3 O3_Script_agentic_ais.py --pr {pr} --model {model}\n")

#Create a text file and write all the lines to it
with open("run_all.txt","w") as f:
    f.writelines(lines)

In [ ]:
####################################################################################################
# Rerunning for NON empty patches
####################################################################################################
sampled_prs=pd.read_csv("sampled_prs.csv")
sampled_prs_ids=[str(id) for id in sampled_prs['PR Id']]
non_empty_patches_list=[patch for patch in sampled_prs_ids if patch not in empty_patches_list]
lines=[]
for app in os.listdir(pjoin("repair-benchmark","apps")):
    pr_dirs=[pjoin("repair-benchmark","apps",app,"issues",folder) for folder in (os.listdir(pjoin("repair-benchmark","apps",app,"issues"))) if not folder.endswith("_private")]
    for pr in pr_dirs:
        for model in ["claude","codex --codex-model gpt-6-astra", "qwen"]:
            with open(pjoin("repair-benchmark","apps",app,"issues",pr.split("/")[-1],f"run_{model}.txt"),"w") as f:
                if pr.split("_")[-1] in non_empty_patches_list and model in empty_patches_agent:
                    lines.append(f"python3 O3_Script_agentic_ais.py --pr {pr} --model {model}\n")

#Create a text file and write all the lines to it
with open("run_all.txt","w") as f:
    f.writelines(lines)


In [10]:
################################################################
# Create run_all.sh file to run all the commands in run_all.txt
################################################################
header = '''#!/bin/bash

TOTAL_START=$(date +%s%3N)
CMD_NUM=0
TOTAL_CMDS={total_cmds}

run_cmd() {{
    CMD_NUM=$((CMD_NUM + 1))
    local CMD="$*"
    echo ""
    echo "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"
    echo "  [$CMD_NUM/$TOTAL_CMDS] Running: $CMD"
    echo "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"
    local START=$(date +%s%3N)

    eval "$CMD"
    local EXIT_CODE=$?

    local END=$(date +%s%3N)
    local ELAPSED=$(echo "scale=2; ($END - $START) / 1000" | bc)
    echo ""
    echo "  Total elapsed time: ${{ELAPSED}} seconds  (exit code: $EXIT_CODE)"

    if [ $EXIT_CODE -ne 0 ]; then
        echo "  ⚠️  Command exited with non-zero status: $EXIT_CODE"
    fi
}}

'''

footer = '''

TOTAL_END=$(date +%s%3N)
TOTAL_ELAPSED=$(echo "scale=2; ($TOTAL_END - $TOTAL_START) / 1000" | bc)
echo ""
echo "════════════════════════════════════════════════════════════════════"
echo "  ✅ All $TOTAL_CMDS commands completed."
echo "  Total elapsed time: ${TOTAL_ELAPSED} seconds"
echo "════════════════════════════════════════════════════════════════════"

# Desktop notification (falls back to terminal bell if notify-send unavailable)
if command -v notify-send &> /dev/null; then
    notify-send "Benchmark finished" "All $TOTAL_CMDS commands completed at $(date). Elapsed: ${TOTAL_ELAPSED}s"
else
    echo -e "\\a"
fi
'''

# Read commands from run_all.txt instead of regenerating them
with open("run_all.txt", "r") as f:
    raw_lines = [line.strip() for line in f if line.strip()]

commands = [f'run_cmd {line}' for line in raw_lines]

total_cmds = len(commands)

script_content = header.format(total_cmds=total_cmds) + "\n".join(commands) + footer

with open("run_all.sh", "w") as f:
    f.write(script_content)

print(f"Wrote {total_cmds} commands to run_all.sh")

Wrote 3 commands to run_all.sh
